# E2 — Multi-head self-attention và tokenization

So sánh MSA viết từ các phép toán nền tảng với `nn.MultiheadAttention`, đồng thời khảo sát patch/row/CNN tokenization trên cùng Fashion-MNIST split.


## 1. Môi trường


### 1.1. Cài đặt thư viện

**Mục tiêu:** Kiểm tra các dependency nền tảng và chỉ cài những gói còn thiếu trong runtime.

**Kết quả:** Môi trường có đủ thư viện để xử lý dữ liệu, huấn luyện attention và trực quan hóa kết quả.


In [1]:
import importlib.util, subprocess, sys

required = {
    "torch": "torch>=2.2,<3",
    "torchvision": "torchvision>=0.17,<1",
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.1,<4",
    "matplotlib": "matplotlib>=3.8,<4",
    "seaborn": "seaborn>=0.13,<1",
    "sklearn": "scikit-learn>=1.4,<2",
    "tqdm": "tqdm>=4.66,<5",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready")


Dependencies ready


### 1.2. Import và chọn thiết bị

**Mục tiêu:** Nạp thư viện dùng chung, chọn CPU/GPU và cấu hình mixed precision theo thiết bị.

**Kết quả:** Tạo `DEVICE`, `AMP_ENABLED` và các namespace được dùng ở các cell sau.


In [2]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
print("device:", DEVICE)


device: cuda


## 2. Cấu hình


### 2.1. Khai báo cấu hình chạy

**Mục tiêu:** Thiết lập đường dẫn, seed, batch size, epoch, chế độ chạy và các experiment cần thực hiện.

**Kết quả:** Tạo `CONFIG`; chế độ `smoke` tự giảm ngân sách để kiểm tra nhanh.


In [3]:
def default_root(kind):
    env_name = "CO5085_DATA_ROOT" if kind == "data" else "CO5085_OUTPUT_ROOT"
    if os.getenv(env_name):
        return Path(os.environ[env_name])
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working") / ("data" if kind == "data" else "artifacts")
    if Path("/content").exists():
        return Path("/content") / ("data" if kind == "data" else "artifacts")
    return Path("./data" if kind == "data" else "./artifacts")


CONFIG = {
    "data_root": str(default_root("data")),
    "output_root": str(default_root("output")),
    # "run_mode": os.getenv("CO5085_RUN_MODE", "smoke"),
    "run_mode": os.getenv("CO5085_RUN_MODE", "full"),
    "experiment_ids": [],  # Empty means all experiments for this notebook.
    "seed": 42,
    "batch_size": 128,
    "epochs": 20,
    "resume_from": None,  # None, "last", or a checkpoint path.
    "evaluate_only": os.getenv("CO5085_EVALUATE_ONLY", "0") == "1",
    "num_workers": 2,
}
assert CONFIG["run_mode"] in {"smoke", "full"}
if CONFIG["run_mode"] == "smoke":
    CONFIG["epochs"] = 1
    CONFIG["num_workers"] = 0
print(json.dumps(CONFIG, indent=2))


{
  "data_root": "/content/data",
  "output_root": "/content/artifacts",
  "run_mode": "full",
  "experiment_ids": [],
  "seed": 42,
  "batch_size": 128,
  "epochs": 20,
  "resume_from": null,
  "evaluate_only": false,
  "num_workers": 2
}


### 2.2. Khởi tạo tiện ích thực nghiệm

**Mục tiêu:** Định nghĩa hàm tái lập, checkpoint, resume, training loop, benchmark và biểu đồ dùng chung.

**Kết quả:** Cung cấp hạ tầng cho toàn bộ sáu cấu hình E2 mà chưa bắt đầu train.


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def environment_info():
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "sklearn": sklearn.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }


def json_dump(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, default=str)


def atomic_torch_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**20
    return {"parameters": total, "trainable_parameters": trainable, "weight_size_mb": size_mb}


def split_digest(train_idx, val_idx):
    digest = hashlib.sha256()
    digest.update(np.asarray(train_idx, dtype=np.int64).tobytes())
    digest.update(np.asarray(val_idx, dtype=np.int64).tobytes())
    return digest.hexdigest()


def fashion_split(dataset, data_root, seed=42):
    split_path = Path(data_root) / "fashion_mnist_split_seed42.npz"
    labels = np.asarray(dataset.targets)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=6000, random_state=seed)
    expected_train, expected_val = next(splitter.split(np.zeros(len(labels)), labels))
    if split_path.exists():
        saved = np.load(split_path)
        train_idx, val_idx = saved["train_idx"], saved["val_idx"]
        if not (np.array_equal(train_idx, expected_train) and np.array_equal(val_idx, expected_val)):
            raise RuntimeError(f"Split file does not match seed={seed}: {split_path}")
    else:
        split_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(split_path, train_idx=expected_train, val_idx=expected_val)
        train_idx, val_idx = expected_train, expected_val
    assert len(train_idx) == 54000 and len(val_idx) == 6000
    assert not set(train_idx).intersection(set(val_idx))
    return train_idx, val_idx, split_digest(train_idx, val_idx)


def classification_epoch(model, loader, criterion, optimizer=None, scaler=None,
                         max_batches=None, grad_clip=None, grad_accum=1, frozen_eval=None):
    training = optimizer is not None
    model.train(training)
    if training and frozen_eval is not None:
        frozen_eval(model)
    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss = total_correct = total_items = 0
    accumulation_count = 0
    start = time.perf_counter()
    context = torch.enable_grad if training else torch.no_grad
    with context():
        for batch_index, (images, labels) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits, labels)
                backward_loss = loss / grad_accum
            if training:
                scaler.scale(backward_loss).backward()
                accumulation_count += 1
                should_step = accumulation_count == grad_accum
                if should_step:
                    if grad_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    accumulation_count = 0
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_items += labels.size(0)
        if training and accumulation_count:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    elapsed = time.perf_counter() - start
    return {"loss": total_loss / total_items, "accuracy": total_correct / total_items,
            "items": total_items, "seconds": elapsed}


@torch.no_grad()
def predict_classifier(model, loader, max_batches=None):
    model.eval()
    labels_all, predictions_all, probabilities_all = [], [], []
    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        logits = model(images.to(DEVICE))
        probabilities = logits.softmax(1).cpu()
        labels_all.append(labels.cpu())
        predictions_all.append(probabilities.argmax(1))
        probabilities_all.append(probabilities)
    return (torch.cat(labels_all).numpy(), torch.cat(predictions_all).numpy(),
            torch.cat(probabilities_all).numpy())


def resolve_resume(run_dir, resume_from):
    if resume_from is None:
        return None
    if resume_from == "last":
        candidate = Path(run_dir) / "last.pt"
    else:
        candidate = Path(resume_from)
    return candidate if candidate.exists() else None


def fit_classifier(model, loaders, optimizer, scheduler, run_dir, run_config,
                   grad_clip=None, grad_accum=1, frozen_eval=None):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    json_dump(run_config, run_dir / "config.json")
    json_dump(environment_info(), run_dir / "environment.json")
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    start_epoch, best_accuracy, best_loss, history = 0, -1.0, float("inf"), []
    if run_config.get("evaluate_only"):
        evaluation_path = run_dir / "best.pt"
        if not evaluation_path.exists():
            raise FileNotFoundError(f"evaluate_only requires {evaluation_path}")
        checkpoint = torch.load(evaluation_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        start_epoch = run_config["epochs"]
        best_accuracy, best_loss = checkpoint["best_accuracy"], checkpoint["best_loss"]
        history_path = run_dir / "history.csv"
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")
    else:
        resume_path = resolve_resume(run_dir, run_config.get("resume_from"))
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False) if resume_path is not None else None
    if not run_config.get("evaluate_only") and checkpoint is not None:
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        if scheduler is not None and checkpoint.get("scheduler") is not None:
            scheduler.load_state_dict(checkpoint["scheduler"])
        if checkpoint.get("scaler") is not None:
            scaler.load_state_dict(checkpoint["scaler"])
        restore_rng_state(checkpoint["rng_state"])
        start_epoch = checkpoint["epoch"] + 1
        best_accuracy, best_loss = checkpoint["best_accuracy"], checkpoint["best_loss"]
        history_path = run_dir / "history.csv"
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(start_epoch, run_config["epochs"]):
        train_stats = classification_epoch(
            model, loaders["train"], criterion, optimizer, scaler,
            max_batches=run_config.get("max_train_batches"), grad_clip=grad_clip,
            grad_accum=grad_accum, frozen_eval=frozen_eval,
        )
        val_stats = classification_epoch(
            model, loaders["val"], criterion,
            max_batches=run_config.get("max_eval_batches"),
        )
        if scheduler is not None:
            scheduler.step()
        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"], "train_accuracy": train_stats["accuracy"],
            "val_loss": val_stats["loss"], "val_accuracy": val_stats["accuracy"],
            "epoch_seconds": train_stats["seconds"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        improved = (val_stats["accuracy"] > best_accuracy or
                    (val_stats["accuracy"] == best_accuracy and val_stats["loss"] < best_loss))
        if improved:
            best_accuracy, best_loss = val_stats["accuracy"], val_stats["loss"]
        state = {
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "scaler": scaler.state_dict(), "epoch": epoch,
            "best_accuracy": best_accuracy, "best_loss": best_loss,
            "rng_state": rng_state(), "config": run_config,
        }
        atomic_torch_save(state, run_dir / "last.pt")
        if improved:
            atomic_torch_save(state, run_dir / "best.pt")
        print(f"epoch={epoch + 1}/{run_config['epochs']} train={train_stats['accuracy']:.4f} val={val_stats['accuracy']:.4f}")

    best = torch.load(run_dir / "best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model"])
    val_stats = classification_epoch(model, loaders["val"], criterion,
                                     max_batches=run_config.get("max_eval_batches"))
    test_stats = classification_epoch(model, loaders["test"], criterion,
                                      max_batches=run_config.get("max_eval_batches"))
    metrics = {
        "run_mode": run_config["run_mode"], "best_epoch": best["epoch"],
        "val_loss": val_stats["loss"], "val_accuracy": val_stats["accuracy"],
        "test_loss": test_stats["loss"], "test_accuracy": test_stats["accuracy"],
        "peak_gpu_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                               if torch.cuda.is_available() else 0.0),
        **count_parameters(model),
    }
    y_true, y_pred, probabilities = predict_classifier(
        model, loaders["test"], max_batches=run_config.get("max_eval_batches"))
    np.savez_compressed(run_dir / "predictions.npz", y_true=y_true, y_pred=y_pred,
                        probabilities=probabilities)
    json_dump(metrics, run_dir / "metrics.json")
    return metrics, pd.DataFrame(history), (y_true, y_pred, probabilities)


@torch.no_grad()
def benchmark_model(model, sample, repeats=30, warmup=5):
    model.eval()
    sample = sample.to(DEVICE)
    for _ in range(warmup):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for _ in range(repeats):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        "latency_ms_per_image": 1000 * elapsed / (repeats * len(sample)),
        "benchmark_batch_size": len(sample),
        "benchmark_peak_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                                     if torch.cuda.is_available() else 0.0),
    }


def plot_history(history, path):
    frame = pd.DataFrame(history)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(frame.epoch, frame.train_loss, label="train")
    axes[0].plot(frame.epoch, frame.val_loss, label="validation")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].plot(frame.epoch, frame.train_accuracy, label="train")
    axes[1].plot(frame.epoch, frame.val_accuracy, label="validation")
    axes[1].set(title="Accuracy", xlabel="Epoch")
    for axis in axes:
        axis.legend(); axis.grid(alpha=.2)
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def plot_confusion(y_true, y_pred, class_names, path, normalize=True):
    matrix = confusion_matrix(y_true, y_pred, normalize="true" if normalize else None)
    size = max(7, min(20, len(class_names) * .22))
    figure, axis = plt.subplots(figsize=(size, size))
    sns.heatmap(matrix, cmap="Blues", xticklabels=class_names, yticklabels=class_names,
                annot=len(class_names) <= 10, fmt=".2f" if normalize else "d", ax=axis)
    axis.set(xlabel="Predicted", ylabel="True", title="Confusion matrix")
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def completed_metrics(run_dir, run_mode):
    path = Path(run_dir) / "metrics.json"
    if not path.exists():
        return None
    with path.open(encoding="utf-8") as handle:
        metrics = json.load(handle)
    return metrics if metrics.get("run_mode") == run_mode else None


seed_everything(CONFIG["seed"])
Path(CONFIG["data_root"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["output_root"]).mkdir(parents=True, exist_ok=True)


## 3. Dữ liệu


### 3.1. Chuẩn bị Fashion-MNIST

**Mục tiêu:** Dùng đúng split phân tầng và thống kê chuẩn hóa của E1/E3, sau đó dựng DataLoader cho train/validation/test.

**Kết quả:** Tạo các dataset, `loaders`, `CLASS_NAMES` và `SPLIT_HASH` để đối chiếu giữa notebook.


In [5]:
raw_train = datasets.FashionMNIST(CONFIG["data_root"], train=True, download=True)
mean = (raw_train.data.float() / 255.0).mean().item()
std = (raw_train.data.float() / 255.0).std().item()
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((mean,), (std,))])
train_full = datasets.FashionMNIST(CONFIG["data_root"], train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(CONFIG["data_root"], train=False, transform=transform, download=True)
train_idx, val_idx, SPLIT_HASH = fashion_split(raw_train, CONFIG["data_root"], CONFIG["seed"])
train_dataset, val_dataset = Subset(train_full, train_idx), Subset(train_full, val_idx)

loaders = {
    "train": DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
                        num_workers=CONFIG["num_workers"], pin_memory=AMP_ENABLED),
    "val": DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
                      num_workers=CONFIG["num_workers"], pin_memory=AMP_ENABLED),
    "test": DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
                       num_workers=CONFIG["num_workers"], pin_memory=AMP_ENABLED),
}
CLASS_NAMES = list(raw_train.classes)
print({"mean": mean, "std": std, "split_hash": SPLIT_HASH,
       "sizes": [len(train_dataset), len(val_dataset), len(test_dataset)]})


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 167kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.14MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 26.9MB/s]


{'mean': 0.2860405743122101, 'std': 0.3530242443084717, 'split_hash': 'bbadd2fc92455ce7f7ca22890516a1f02c47a663679ec246709d7cf0cfdcb075', 'sizes': [54000, 6000, 10000]}


## 4. Tokenizer, attention và mô hình encoder


### 4.1. Xây dựng tokenizer và encoder

**Mục tiêu:** Cài đặt patch, row, CNN-stem tokenizer; MSA tự viết; MSA PyTorch; encoder hai block và classifier dùng mean pooling.

**Kết quả:** Tạo sáu experiment ID từ hai backend attention và ba tokenizer, cùng hàm đồng bộ khởi tạo giữa hai backend.


In [6]:
class PatchTokenizer(nn.Module):
    token_count = 49
    def __init__(self, embed_dim=64):
        super().__init__(); self.projection = nn.Linear(16, embed_dim)
    def forward(self, images):
        patches = F.unfold(images, kernel_size=4, stride=4).transpose(1, 2)
        return self.projection(patches)


class RowTokenizer(nn.Module):
    token_count = 28
    def __init__(self, embed_dim=64):
        super().__init__(); self.projection = nn.Linear(28, embed_dim)
    def forward(self, images):
        return self.projection(images.squeeze(1))


class CNNTokenizer(nn.Module):
    token_count = 49
    def __init__(self, embed_dim=64):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
                                  nn.Conv2d(32, embed_dim, 3, stride=2, padding=1), nn.ReLU())
    def forward(self, images):
        return self.stem(images).flatten(2).transpose(1, 2)


class ScratchMSA(nn.Module):
    def __init__(self, embed_dim=64, heads=4, dropout=0.0):
        super().__init__()
        assert embed_dim % heads == 0
        self.heads, self.head_dim = heads, embed_dim // heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, inputs):
        batch, tokens, width = inputs.shape
        qkv = self.qkv(inputs).reshape(batch, tokens, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        attention = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attention = self.dropout(attention.softmax(-1))
        context = (attention @ v).transpose(1, 2).reshape(batch, tokens, width)
        return self.out(context)


class TorchMSA(nn.Module):
    def __init__(self, embed_dim=64, heads=4, dropout=0.0):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, heads, dropout=dropout, batch_first=True)
    def forward(self, inputs):
        return self.mha(inputs, inputs, inputs, need_weights=False)[0]


class EncoderBlock(nn.Module):
    def __init__(self, backend, embed_dim=64, heads=4, ffn_dim=128):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(embed_dim), nn.LayerNorm(embed_dim)
        self.attn = ScratchMSA(embed_dim, heads) if backend == "scratch" else TorchMSA(embed_dim, heads)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, embed_dim))
    def forward(self, inputs):
        hidden = inputs + self.attn(self.norm1(inputs))
        return hidden + self.ffn(self.norm2(hidden))


TOKENIZERS = {"patch": PatchTokenizer, "row": RowTokenizer, "cnn": CNNTokenizer}


class ImageAttentionClassifier(nn.Module):
    def __init__(self, backend, tokenizer_name, embed_dim=64, heads=4, depth=2, ffn_dim=128):
        super().__init__()
        self.backend, self.tokenizer_name = backend, tokenizer_name
        self.tokenizer = TOKENIZERS[tokenizer_name](embed_dim)
        self.position = nn.Parameter(torch.zeros(1, self.tokenizer.token_count, embed_dim))
        nn.init.trunc_normal_(self.position, std=.02)
        self.blocks = nn.ModuleList([EncoderBlock(backend, embed_dim, heads, ffn_dim) for _ in range(depth)])
        self.norm, self.head = nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 10)
    def forward(self, images):
        tokens = self.tokenizer(images) + self.position
        for block in self.blocks:
            tokens = block(tokens)
        return self.head(self.norm(tokens).mean(1))


def copy_attention(scratch, torch_msa):
    torch_msa.mha.in_proj_weight.data.copy_(scratch.qkv.weight.data)
    torch_msa.mha.in_proj_bias.data.copy_(scratch.qkv.bias.data)
    torch_msa.mha.out_proj.weight.data.copy_(scratch.out.weight.data)
    torch_msa.mha.out_proj.bias.data.copy_(scratch.out.bias.data)


def make_model(backend, tokenizer_name):
    seed_everything(CONFIG["seed"])
    reference = ImageAttentionClassifier("scratch", tokenizer_name)
    if backend == "scratch":
        return reference
    target = ImageAttentionClassifier("torch", tokenizer_name)
    target.tokenizer.load_state_dict(reference.tokenizer.state_dict())
    target.position.data.copy_(reference.position.data)
    target.norm.load_state_dict(reference.norm.state_dict())
    target.head.load_state_dict(reference.head.state_dict())
    for source_block, target_block in zip(reference.blocks, target.blocks):
        target_block.norm1.load_state_dict(source_block.norm1.state_dict())
        target_block.norm2.load_state_dict(source_block.norm2.state_dict())
        target_block.ffn.load_state_dict(source_block.ffn.state_dict())
        copy_attention(source_block.attn, target_block.attn)
    return target


EXPERIMENT_IDS = [f"{backend}_{tokenizer}" for backend in ("scratch", "torch")
                  for tokenizer in ("patch", "row", "cnn")]
selected = CONFIG["experiment_ids"] or EXPERIMENT_IDS
unknown = set(selected) - set(EXPERIMENT_IDS)
if unknown: raise ValueError(f"Unknown experiment IDs: {sorted(unknown)}")
print("Selected:", selected)


Selected: ['scratch_patch', 'scratch_row', 'scratch_cnn', 'torch_patch', 'torch_row', 'torch_cnn']


## 5. Kiểm chứng tương đương số học


### 5.1. So sánh MSA và kiểm tra pipeline

**Mục tiêu:** Sao chép trọng số giữa hai MSA, so output/gradient trên CPU FP32, kiểm tra shape tokenizer và overfit một batch.

**Kết quả:** Các assertion xác nhận tolerance `atol=1e-5`, `rtol=1e-4`, shape token đúng và loss có giảm.


In [7]:
def verify_attention_equivalence():
    seed_everything(CONFIG["seed"])
    scratch, builtin = ScratchMSA().cpu(), TorchMSA().cpu()
    copy_attention(scratch, builtin)
    x1 = torch.randn(3, 7, 64, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    y1, y2 = scratch(x1), builtin(x2)
    torch.testing.assert_close(y1, y2, atol=1e-5, rtol=1e-4)
    weights = torch.randn_like(y1)
    (y1 * weights).sum().backward(); (y2 * weights).sum().backward()
    torch.testing.assert_close(x1.grad, x2.grad, atol=1e-5, rtol=1e-4)
    torch.testing.assert_close(scratch.qkv.weight.grad, builtin.mha.in_proj_weight.grad, atol=1e-5, rtol=1e-4)
    torch.testing.assert_close(scratch.qkv.bias.grad, builtin.mha.in_proj_bias.grad, atol=1e-5, rtol=1e-4)
    torch.testing.assert_close(scratch.out.weight.grad, builtin.mha.out_proj.weight.grad, atol=1e-5, rtol=1e-4)
    print("MSA output, input gradients and parameter gradients match")


verify_attention_equivalence()
for tokenizer_name in TOKENIZERS:
    sample = torch.randn(2, 1, 28, 28)
    assert TOKENIZERS[tokenizer_name]()(sample).shape[1] == TOKENIZERS[tokenizer_name].token_count
print("Tokenizer shapes are valid")

probe = make_model("scratch", "patch").to(DEVICE)
images, labels = next(iter(loaders["train"])); images, labels = images[:32].to(DEVICE), labels[:32].to(DEVICE)
probe_optimizer = torch.optim.AdamW(probe.parameters(), lr=3e-3)
probe_initial = None
for _ in range(10 if CONFIG["run_mode"] == "smoke" else 30):
    probe_optimizer.zero_grad(set_to_none=True)
    probe_loss = F.cross_entropy(probe(images), labels)
    if probe_initial is None: probe_initial = probe_loss.item()
    probe_loss.backward(); probe_optimizer.step()
assert torch.isfinite(probe_loss) and probe_loss.item() < probe_initial
print("E2 one-batch overfit:", probe_initial, "->", probe_loss.item())
del probe, probe_optimizer
if torch.cuda.is_available(): torch.cuda.empty_cache()


MSA output, input gradients and parameter gradients match
Tokenizer shapes are valid
E2 one-batch overfit: 2.4051413536071777 -> 0.04063470661640167


## 6. Huấn luyện và đánh giá ma trận 2 × 3


### 6.1. Chạy sáu cấu hình E2

**Mục tiêu:** Huấn luyện từng cặp backend–tokenizer với cùng seed và ngân sách, sau đó đo accuracy, latency, memory và số token.

**Kết quả:** Lưu checkpoint, metrics, predictions, curves, confusion matrix và bảng tổng hợp `summary.csv`.

**Lưu ý:** Run đã hoàn tất được bỏ qua trừ khi bật chế độ đánh giá lại checkpoint.


In [8]:
summary = []
OUTPUT_BASE = Path(CONFIG["output_root"]) / "E2" / CONFIG["run_mode"]
for experiment_id in selected:
    backend, tokenizer_name = experiment_id.split("_", 1)
    run_dir = OUTPUT_BASE / experiment_id
    existing = completed_metrics(run_dir, CONFIG["run_mode"])
    if existing is not None and not CONFIG["evaluate_only"]:
        summary.append({"experiment_id": experiment_id, **existing}); continue
    model = make_model(backend, tokenizer_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    run_config = {
        **CONFIG, "assignment": "E2", "experiment_id": experiment_id,
        "backend": backend, "tokenizer": tokenizer_name,
        "token_count": model.tokenizer.token_count, "split_hash": SPLIT_HASH,
        "max_train_batches": 4 if CONFIG["run_mode"] == "smoke" else None,
        "max_eval_batches": 2 if CONFIG["run_mode"] == "smoke" else None,
    }
    metrics, history, predictions = fit_classifier(model, loaders, optimizer, None, run_dir, run_config)
    sample_images, _ = next(iter(loaders["test"]))
    metrics.update(benchmark_model(model, sample_images[:32], repeats=5 if CONFIG["run_mode"] == "smoke" else 30))
    metrics["token_count"] = model.tokenizer.token_count
    json_dump(metrics, run_dir / "metrics.json")
    plot_history(history, run_dir / "curves.png")
    plot_confusion(predictions[0], predictions[1], CLASS_NAMES, run_dir / "confusion_matrix.png")
    summary.append({"experiment_id": experiment_id, **metrics})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

summary_frame = pd.DataFrame(summary).sort_values("experiment_id")
summary_frame.to_csv(OUTPUT_BASE / "summary.csv", index=False)
display(summary_frame)


epoch=1/20 train=0.6895 val=0.8087
epoch=2/20 train=0.8211 val=0.8333
epoch=3/20 train=0.8435 val=0.8458
epoch=4/20 train=0.8576 val=0.8415
epoch=5/20 train=0.8654 val=0.8662
epoch=6/20 train=0.8718 val=0.8680
epoch=7/20 train=0.8761 val=0.8733
epoch=8/20 train=0.8816 val=0.8842
epoch=9/20 train=0.8861 val=0.8762
epoch=10/20 train=0.8902 val=0.8758
epoch=11/20 train=0.8922 val=0.8780
epoch=12/20 train=0.8960 val=0.8875
epoch=13/20 train=0.9007 val=0.8822
epoch=14/20 train=0.9025 val=0.8828
epoch=15/20 train=0.9061 val=0.8873
epoch=16/20 train=0.9096 val=0.8840
epoch=17/20 train=0.9118 val=0.8913
epoch=18/20 train=0.9144 val=0.8912
epoch=19/20 train=0.9173 val=0.8822
epoch=20/20 train=0.9182 val=0.8863
epoch=1/20 train=0.7912 val=0.8543
epoch=2/20 train=0.8545 val=0.8660
epoch=3/20 train=0.8706 val=0.8792
epoch=4/20 train=0.8813 val=0.8802
epoch=5/20 train=0.8879 val=0.8905
epoch=6/20 train=0.8931 val=0.8937
epoch=7/20 train=0.8979 val=0.8880
epoch=8/20 train=0.9018 val=0.8920
epoch=9/2

,experiment_id,run_mode,best_epoch,val_loss,val_accuracy,test_loss,test_accuracy,peak_gpu_memory_mb,parameters,trainable_parameters,weight_size_mb,latency_ms_per_image,benchmark_batch_size,benchmark_peak_memory_mb,token_count
2,scratch_cnn,full,17,0.265749,0.915000,0.306578,0.9020,68.359375,89674,89674,0.342079,0.048978,32,23.559570,49
0,scratch_patch,full,16,0.311959,0.891333,0.346668,0.8809,64.339844,71946,71946,0.274452,0.042900,32,23.354492,49
1,scratch_row,full,16,0.288205,0.905167,0.327272,0.8923,41.274902,71370,71370,0.272255,0.035891,32,21.002441,28
5,torch_cnn,full,14,0.254695,0.912000,0.295165,0.8975,54.185547,89674,89674,0.342079,0.038101,32,21.980469,49
3,torch_patch,full,17,0.292749,0.895000,0.330479,0.8804,51.741211,71946,71946,0.274452,0.034335,32,21.775391,49
4,torch_row,full,13,0.290417,0.903333,0.340879,0.8834,41.568848,71370,71370,0.272255,0.027165,32,20.619629,28


## 7. Câu hỏi phân tích

1. Sau khi đồng bộ trọng số, sai số output/gradient có nằm trong tolerance đã công bố không?
2. Token count liên hệ ra sao với thời gian, bộ nhớ và accuracy?
3. CNN-stem có lợi thế gì và đã thêm inductive bias/tham số nào vào phép so sánh?
4. Backend scratch và PyTorch có accuracy tương đương khi khởi tạo tương ứng không? Nếu không, kiểm tra lại randomness và kernel trước khi kết luận.


## 8. Tải xuống các tệp kết quả thí nghiệm

Đóng gói các tệp kết quả từ lần chạy chính thức với chế độ `full` của thí nghiệm E2 thành một tệp ZIP để lưu trữ trên máy tính cá nhân, phân tích kết quả và chuẩn bị báo cáo.

Tệp ZIP bao gồm các tệp kết quả hiện có trong thư mục `artifacts/E2/full/` của cả sáu cấu hình thí nghiệm:

- **Các checkpoint của mô hình:** `best.pt`, `last.pt`.
- **Các chỉ số đánh giá:** `metrics.json`, `summary.csv`.
- **Lịch sử huấn luyện:** `history.csv`.
- **Các hình ảnh trực quan hóa:** `curves.png`, `confusion_matrix.png`.
- **Kết quả dự đoán:** `predictions.npz`.
- **Thông tin cấu hình thí nghiệm:** `config.json`, `environment.json`.

Chỉ các tệp thuộc lần chạy ở chế độ `full` được đưa vào tệp ZIP. Các tệp dữ liệu và chỉ số phân chia tập dữ liệu không được bao gồm. Trước khi tạo tệp ZIP, ô lệnh tải xuống sẽ kiểm tra để bảo đảm cả sáu thí nghiệm đã hoàn thành và mỗi thí nghiệm đều có tệp `metrics.json` ghi nhận kết quả từ chế độ `full`.

Các kết quả từ lần chạy thử nghiệm nhanh ở chế độ `smoke` không được sử dụng trong quá trình phân tích chính thức.

Sau khi giải nén, cấu trúc thư mục được giữ nguyên như sau:

```text
artifacts/
└── E2/
    └── full/
        ├── summary.csv
        ├── scratch_patch/
        ├── scratch_row/
        ├── scratch_cnn/
        ├── torch_patch/
        ├── torch_row/
        └── torch_cnn/
```

In [9]:
from pathlib import Path
import json
import zipfile
from google.colab import files

output_root = Path(CONFIG["output_root"])
results_dir = output_root / "E2" / "full"
zip_path = Path("/content/E2_full_results.zip")

# Ensure this notebook is configured for the official full run.
if CONFIG["run_mode"] != "full":
    raise RuntimeError(
        f"Current run_mode is {CONFIG['run_mode']!r}, not 'full'. "
        "Configure run_mode='full' and complete the full experiments first."
    )

# All six E2 experiment configurations must be present.
expected_experiments = [
    "scratch_patch",
    "scratch_row",
    "scratch_cnn",
    "torch_patch",
    "torch_row",
    "torch_cnn",
]

if not results_dir.exists():
    raise FileNotFoundError(
        f"Full-run directory not found: {results_dir}. "
        "Run the full experiments before downloading."
    )

missing = []
invalid = []

for experiment_id in expected_experiments:
    metrics_path = results_dir / experiment_id / "metrics.json"

    if not metrics_path.exists():
        missing.append(experiment_id)
        continue

    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)

    if metrics.get("run_mode") != "full":
        invalid.append(experiment_id)

if missing:
    raise RuntimeError(
        "Missing metrics.json for: "
        + ", ".join(missing)
        + ". Complete these experiments before downloading."
    )

if invalid:
    raise RuntimeError(
        "The following experiments do not contain full-run metrics: "
        + ", ".join(invalid)
    )

summary_path = results_dir / "summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(
        f"Missing summary.csv: {summary_path}. "
        "Check that the experiment summary cell completed successfully."
    )

# Package the entire E2/full directory.
# ZIP structure: artifacts/E2/full/...
with zipfile.ZipFile(
    zip_path, mode="w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in results_dir.rglob("*"):
        if path.is_file():
            archive_path = (
                Path("artifacts") / "E2" / "full"
                / path.relative_to(results_dir)
            )
            zf.write(path, archive_path)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")
print(f"Experiments verified: {len(expected_experiments)}/6")

files.download(str(zip_path))

Created: /content/E2_full_results.zip
Size: 13.35 MB
Experiments verified: 6/6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>